# Ball Court Multi-Agent RL Demo

This notebook demonstrates the Ball Court environment and shows how to:
1. Create and interact with the environment
2. Train agents using PPO
3. Visualize gameplay
4. Evaluate trained agents

In [ ]:
# Import required libraries
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML
import matplotlib.animation as animation

from envs.ball_court import BallCourtEnv, BallCourtEnvSingleAgent
from agents.ppo_agent import PPOAgent, PPOConfig

## 1. Environment Overview

Let's create and explore the Ball Court environment.

In [ ]:
# Create the environment
env = BallCourtEnv(render_mode=None)

# Print environment info
print("Environment Information:")
print(f"  Observation Space: {env.observation_space}")
print(f"  Action Space: {env.action_space}")
print(f"  Max Steps per Episode: {env.max_steps}")

In [ ]:
# Reset the environment
observations, info = env.reset(seed=42)

print("Initial Observations:")
print(f"  Agent 1 observation shape: {observations['agent_1'].shape}")
print(f"  Agent 1 observation: {observations['agent_1']}")
print(f"\n  Agent 2 observation shape: {observations['agent_2'].shape}")
print(f"  Agent 2 observation: {observations['agent_2']}")
print(f"\nInfo: {info}")

## 2. Random Agent Play

Let's see what happens when both agents take random actions.

In [ ]:
# Run a few random episodes
def run_random_episode(env, max_steps=1000):
    """Run an episode with random actions."""
    observations, _ = env.reset()
    
    states = {
        'agent1': [],
        'agent2': [],
        'ball': []
    }
    total_reward = 0
    
    for step in range(max_steps):
        # Random actions for both agents
        actions = {
            'agent_1': env.action_space.sample(),
            'agent_2': env.action_space.sample()
        }
        
        observations, reward, done1, done2, info = env.step(actions)
        total_reward += reward
        
        # Store state for visualization
        states['agent1'].append(info['agent1_position'].copy())
        states['agent2'].append(info['agent2_position'].copy())
        states['ball'].append(info['ball_position'].copy())
        
        if done1 or done2:
            break
    
    return states, total_reward, step + 1, info

# Run a test episode
states, reward, length, final_info = run_random_episode(env)
print(f"Episode completed in {length} steps")
print(f"Total reward: {reward:.2f}")
print(f"Agent 1 lives remaining: {final_info['lives_agent_1']}")
print(f"Agent 2 lives remaining: {final_info['lives_agent_2']}")

## 3. Visualize Gameplay

Create a visualization of the gameplay.

In [ ]:
def plot_gameplay(states, save_path=None):
    """Plot the gameplay trajectory."""
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Convert to arrays
    agent1_pos = np.array(states['agent1'])
    agent2_pos = np.array(states['agent2'])
    ball_pos = np.array(states['ball'])
    
    # Plot court
    ax.set_xlim(-2, 2)
    ax.set_ylim(-1, 1)
    ax.axvline(x=-1.5, color='red', linestyle='--', linewidth=2, label='Agent 1 Goal')
    ax.axvline(x=1.5, color='blue', linestyle='--', linewidth=2, label='Agent 2 Goal')
    ax.axhline(y=-0.75, color='gray', linestyle='-', linewidth=1)
    ax.axhline(y=0.75, color='gray', linestyle='-', linewidth=1)
    ax.axvline(x=0, color='gray', linestyle='-', linewidth=1)
    
    # Plot trajectories
    ax.plot(agent1_pos[:, 0], agent1_pos[:, 1], 'r-', alpha=0.5, label='Agent 1 Path')
    ax.plot(agent2_pos[:, 0], agent2_pos[:, 1], 'b-', alpha=0.5, label='Agent 2 Path')
    ax.plot(ball_pos[:, 0], ball_pos[:, 1], 'g-', alpha=0.5, label='Ball Path')
    
    # Plot start positions
    ax.scatter(agent1_pos[0, 0], agent1_pos[0, 1], c='red', s=200, marker='o', zorder=5)
    ax.scatter(agent2_pos[0, 0], agent2_pos[0, 1], c='blue', s=200, marker='o', zorder=5)
    ax.scatter(ball_pos[0, 0], ball_pos[0, 1], c='green', s=100, marker='o', zorder=5)
    
    # Plot end positions
    ax.scatter(agent1_pos[-1, 0], agent1_pos[-1, 1], c='red', s=200, marker='x', zorder=5)
    ax.scatter(agent2_pos[-1, 0], agent2_pos[-1, 1], c='blue', s=200, marker='x', zorder=5)
    ax.scatter(ball_pos[-1, 0], ball_pos[-1, 1], c='green', s=100, marker='x', zorder=5)
    
    ax.set_xlabel('X Position')
    ax.set_ylabel('Y Position')
    ax.set_title('Ball Court Gameplay Trajectory')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path)
        print(f"Saved plot to {save_path}")
    
    plt.show()
    return fig

# Plot the gameplay
fig = plot_gameplay(states)

## 4. Create and Use a PPO Agent

Let's create a PPO agent and see how it performs.

In [ ]:
# Create a PPO agent
config = PPOConfig()
agent = PPOAgent(state_dim=12, action_dim=5, config=config)

print("PPO Agent Created")
print(f"  Device: {agent.device}")
print(f"  Learning rate (actor): {config.lr_actor}")
print(f"  Learning rate (critic): {config.lr_critic}")
print(f"  Gamma: {config.gamma}")

In [ ]:
# Run an episode with the PPO agent against a random opponent
def run_ppo_vs_random(agent, env, max_steps=1000):
    """Run an episode with PPO agent vs random opponent."""
    observations, _ = env.reset()
    
    states = {
        'agent1': [],
        'agent2': [],
        'ball': []
    }
    total_reward = 0
    actions_taken = []
    
    for step in range(max_steps):
        # PPO agent action
        action1, _, _ = agent.select_action(observations['agent_1'], training=False)
        
        # Random opponent action
        action2 = env.action_space.sample()
        
        actions_taken.append(action1)
        
        observations, reward, done1, done2, info = env.step({
            'agent_1': action1,
            'agent_2': action2
        })
        
        total_reward += reward
        
        states['agent1'].append(info['agent1_position'].copy())
        states['agent2'].append(info['agent2_position'].copy())
        states['ball'].append(info['ball_position'].copy())
        
        if done1 or done2:
            break
    
    return states, total_reward, step + 1, info, actions_taken

# Run a test episode
states, reward, length, final_info, actions = run_ppo_vs_random(agent, env)
print(f"Episode completed in {length} steps")
print(f"Total reward: {reward:.2f}")
print(f"Agent 1 lives: {final_info['lives_agent_1']}, Agent 2 lives: {final_info['lives_agent_2']}")

# Action distribution
action_counts = np.bincount(actions, minlength=5)
action_names = ['Forward', 'Backward', 'Punch', 'Left', 'Right']
print("\nAction Distribution:")
for i, (name, count) in enumerate(zip(action_names, action_counts)):
    print(f"  {name}: {count} ({100*count/len(actions):.1f}%)")

## 5. Quick Training Demo

Let's run a short training loop to see the agent learn.

In [ ]:
# Quick training demo
def quick_train(agent, env, num_episodes=50, max_steps=500):
    """Run a quick training loop."""
    rewards = []
    wins = 0
    
    for episode in range(num_episodes):
        observations, _ = env.reset()
        episode_reward = 0
        
        for step in range(max_steps):
            # Agent 1 (PPO)
            action1, _, _ = agent.select_action(observations['agent_1'], training=True)
            
            # Agent 2 (random)
            action2 = env.action_space.sample()
            
            observations, reward, done1, done2, info = env.step({
                'agent_1': action1,
                'agent_2': action2
            })
            
            episode_reward += reward
            
            # Update buffer
            if agent.buffer.rewards:
                agent.buffer.rewards[-1] = reward
                agent.buffer.is_terminals[-1] = done1
            
            if done1 or done2:
                if info['lives_agent_2'] < 5:
                    wins += 1
                break
        
        rewards.append(episode_reward)
        
        # Update every few episodes
        if (episode + 1) % 10 == 0:
            agent.update()
            agent.buffer.clear()
            print(f"Episode {episode+1}/{num_episodes}: "
                  f"Avg Reward (last 10): {np.mean(rewards[-10:]):.2f}, "
                  f"Wins: {wins}")
    
    return rewards, wins

# Run quick training
print("Running quick training demo...")
rewards, total_wins = quick_train(agent, env, num_episodes=50, max_steps=500)
print(f"\nTraining complete! Total wins: {total_wins}/50")

In [ ]:
# Plot training progress
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot rewards
ax1.plot(rewards)
ax1.axhline(y=np.mean(rewards[-10:]), color='r', linestyle='--', 
            label=f'Mean (last 10): {np.mean(rewards[-10:]):.2f}')
ax1.set_xlabel('Episode')
ax1.set_ylabel('Total Reward')
ax1.set_title('Training Rewards')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot cumulative wins
cumulative_wins = np.cumsum([1 if r > 0 else 0 for r in rewards])
ax2.plot(cumulative_wins)
ax2.set_xlabel('Episode')
ax2.set_ylabel('Cumulative Wins')
ax2.set_title('Cumulative Wins Over Training')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Save and Load Agents

Learn how to save and load trained agents.

In [ ]:
import os

# Create models directory
os.makedirs('../outputs/models', exist_ok=True)

# Save the agent
save_path = '../outputs/models/demo_agent.pth'
agent.save(save_path)
print(f"Agent saved to {save_path}")

# Create a new agent and load
new_agent = PPOAgent(state_dim=12, action_dim=5, config=config)
new_agent.load(save_path)
print("New agent loaded successfully!")

## 7. Evaluation

Evaluate the trained agent against different opponent types.

In [ ]:
def evaluate_agent(agent, env, opponent_type='random', num_episodes=20):
    """Evaluate agent against a specific opponent type."""
    wins = 0
    losses = 0
    draws = 0
    total_reward = 0
    
    for _ in range(num_episodes):
        observations, _ = env.reset()
        episode_reward = 0
        
        for _ in range(6000):
            action1, _, _ = agent.select_action(observations['agent_1'], training=False)
            
            if opponent_type == 'random':
                action2 = env.action_space.sample()
            else:
                action2 = 0  # Scripted (stationary)
            
            observations, reward, done1, done2, info = env.step({
                'agent_1': action1,
                'agent_2': action2
            })
            episode_reward += reward
            
            if done1 or done2:
                break
        
        total_reward += episode_reward
        
        if info['lives_agent_2'] < 5:
            wins += 1
        elif info['lives_agent_1'] < 5:
            losses += 1
        else:
            draws += 1
    
    return {
        'wins': wins,
        'losses': losses,
        'draws': draws,
        'win_rate': wins / num_episodes,
        'avg_reward': total_reward / num_episodes
    }

# Evaluate
results = evaluate_agent(agent, env, opponent_type='random', num_episodes=20)
print(f"Evaluation Results (vs random opponent):")
print(f"  Wins: {results['wins']}")
print(f"  Losses: {results['losses']}")
print(f"  Draws: {results['draws']}")
print(f"  Win Rate: {results['win_rate']:.1%}")
print(f"  Average Reward: {results['avg_reward']:.2f}")

## Summary

In this notebook, you learned how to:

1. **Create and interact with the Ball Court environment**
2. **Run random agent play** to understand the environment
3. **Visualize gameplay trajectories** using matplotlib
4. **Create and use a PPO agent** for decision making
5. **Train agents** using the PPO algorithm
6. **Save and load trained agents**
7. **Evaluate agent performance** against different opponents

For more details, see the documentation in `docs/`.

In [ ]:
# Clean up
env.close()
print("Environment closed.")